# GPR Normative Model — Example Usage

This notebook demonstrates the complete workflow:
1. Generate synthetic example data
2. Train GPR models
3. Cross-validate model performance
4. Predict deviations on new subjects
5. Fine-tune on a custom dataset

## 1. Setup and Data Generation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile
import os

# Generate example data
os.system('python ../data/generate_example_data.py --out_dir /tmp/gpr_demo --n_subjects 200 --seed 42')

df = pd.read_csv('/tmp/gpr_demo/example_brain_data.csv')
print(f'Loaded {len(df)} subjects, {len(df.columns)} columns')
df.head()

In [ ]:
# Quick look at age distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['age'], bins=30, edgecolor='black')
axes[0].set_xlabel('Age (years)')
axes[0].set_title('Age Distribution')

df['sex'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Sex Distribution')

df['site'].value_counts().plot(kind='bar', ax=axes[2])
axes[2].set_title('Site Distribution')

plt.tight_layout()
plt.show()

## 2. Training GPR Normative Models

In [ ]:
from gpr_normative.train import train_one_score, build_gpr_pipeline
from gpr_normative.data_utils import load_one_merged_metric_csv, find_score_columns

# Load and prepare data
df = load_one_merged_metric_csv('/tmp/gpr_demo/example_brain_data.csv')
score_cols = find_score_columns(df)
print(f'Found {len(score_cols)} score columns: {score_cols[:5]}...')

conf_num = ['age']
conf_cat = ['sex', 'site', 'breed']

In [ ]:
# Train models for all score columns
out_dir = Path('/tmp/gpr_demo/train_output')

for sc in score_cols[:5]:  # Train first 5 for demo
    result = train_one_score(
        df, sc, conf_num, conf_cat, out_dir,
        nu=2.5, n_restarts_optimizer=3, random_state=42, min_n=10, test_size=0.2,
    )
    if result:
        status = ''
        if 'test_metrics' in result:
            m = result['test_metrics']
            status = f' | Test EXPV={m["expv"]:.4f}, R²={m["r2"]:.4f}'
        print(f'  [{sc}] trained on {result["n_train"]} subjects{status}')

## 3. Cross-Validation Evaluation

In [ ]:
from gpr_normative.evaluate import cross_validate_one_score

cv_results = []
for sc in score_cols[:5]:  # CV first 5 for demo
    r = cross_validate_one_score(
        df, sc, conf_num, conf_cat,
        nu=2.5, n_restarts=2, n_folds=3, random_state=42, min_n=10,
    )
    if r:
        cv_results.append(r)
        print(f'  [{r["score_col"]}] EXPV={r["mean_expv"]:.4f} ± {r["std_expv"]:.4f}, '
              f'MAE={r["mean_mae"]:.4f}, R²={r["mean_r2"]:.4f}')

cv_df = pd.DataFrame(cv_results)
print(f'\nAverage EXPV across {len(cv_df)} ROIs: {cv_df["mean_expv"].mean():.4f}')

## 4. Predicting Deviations on New Subjects

In [ ]:
from gpr_normative.predict import predict_one_metric, list_model_files

# Create some "new" test subjects (with slight age shift to simulate disease)
rng = np.random.default_rng(123)
n_new = 10
test_df = pd.DataFrame({
    'subject_id': [f'new-{i:02d}' for i in range(n_new)],
    'age': rng.uniform(2, 30, n_new),
    'sex': rng.choice(['M', 'F'], n_new),
    'site': 'site_A',
    'breed': 'Macaca_mulatta',
})

# Generate synthetic brain data for these subjects (with slight bias)
for roi in score_cols:
    base_mean = df[roi].mean()
    base_std = df[roi].std()
    # Add slight negative bias to simulate atrophy
    test_df[roi] = rng.normal(base_mean - 0.3 * base_std, base_std, n_new)

# Predict using trained models
model_dir = out_dir / 'models'
pred_dir = Path('/tmp/gpr_demo/pred_output')

for mp in list_model_files(model_dir)[:3]:  # First 3 models for demo
    result = predict_one_metric(
        test_df, mp, pred_dir, conf_num, conf_cat, ['subject_id']
    )
    if result['status'] == 'ok':
        print(f'  [{result["score_col"]}] predicted {result["n_pred"]} subjects')

In [ ]:
# Look at the prediction output for one ROI
pred_csv = pred_dir / 'predictions' / 'V1__predictions.csv'
if pred_csv.exists():
    pred_df = pd.read_csv(pred_csv)
    print(f'Prediction columns: {list(pred_df.columns)}')
    print()
    # Show key columns
    display_cols = ['subject_id', 'age', 'V1', 'V1__mu', 'V1__std', 'V1__z', 'V1__resid']
    display_cols = [c for c in display_cols if c in pred_df.columns]
    display(pred_df[display_cols].head())
    
    # Interpretation
    print('\nInterpretation:')
    print('  - z < -1.96: significant negative deviation (potential atrophy)')
    print('  - z > +1.96: significant positive deviation')
    print(f'  - Mean z-score: {pred_df["V1__z"].mean():.3f}')

## 5. Fine-Tuning on a Custom Dataset

Suppose you have a small new dataset (e.g., 30 subjects from a new site/species).
Instead of training from scratch, you can fine-tune the pretrained models.

In [ ]:
from gpr_normative.fine_tune import fine_tune_one_score

# Simulate a small custom dataset (new breed, fewer subjects)
n_custom = 30
custom_df = pd.DataFrame({
    'subject_id': [f'custom-{i:02d}' for i in range(n_custom)],
    'age': rng.uniform(3, 25, n_custom),
    'sex': rng.choice(['M', 'F'], n_custom),
    'site': 'new_site',
    'breed': 'Macaca_radiata',  # New breed
})

# Generate ROI data with slightly different pattern
for roi in score_cols:
    custom_df[roi] = df[roi].sample(n_custom, random_state=42).values + rng.normal(0.1, 0.15, n_custom)

ft_dir = Path('/tmp/gpr_demo/finetuned_output')

# Fine-tune with frozen length_scale (recommended for small datasets)
for mp in list_model_files(model_dir)[:3]:
    result = fine_tune_one_score(
        custom_df, mp.stem, mp, ft_dir,
        conf_num, conf_cat,
        freeze_length_scale=True, n_restarts=5,
        random_state=42, min_n=5,
    )
    if result:
        print(f'  [{result["score_col"]}] EXPV={result["expv"]:.4f}, R²={result["r2"]:.4f}')
        print(f'    length_scale={result["length_scale"]:.4f}, '
              f'noise_level={result["noise_level"]:.4f}')

## Summary

- **Training**: Uses `ConstantKernel * Matern(nu=2.5) + WhiteKernel` as the GP kernel
- **Prediction**: Outputs `mu` (predicted mean), `std` (uncertainty), `z` (z-score), `resid` (residual)
- **Z-score interpretation**: |z| > 1.96 indicates significant deviation from the normative range (p < 0.05)
- **Fine-tuning**: Adjust `n_restarts` (more = better fit) and `freeze_length_scale` (True for small datasets)